# LegalEagle — Qdrant RAG QA Chain

Notebook 3 — Full RAG pipeline with **Qdrant** vector store (local Docker), metadata filtering, and cited answers.

**Pipeline:** Question → Embed → Qdrant search → top-5 chunks → LLM → cited answer

**Tech:** Qdrant · LangChain · sentence-transformers/all-MiniLM-L6-v2 · flan-t5-base

## 0 — Imports & Setup

In [26]:
import os, re, json, warnings, textwrap
from pathlib import Path
from datetime import datetime
import numpy as np
warnings.filterwarnings("ignore")

# LangChain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

# Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, PointStruct,
    Filter, FieldCondition, MatchValue, Range
)

# HuggingFace LLM
from transformers import pipeline as hf_pipeline

print("All imports OK!")

All imports OK!


## 1 — Start Qdrant via Docker

We run Qdrant locally in Docker. The container exposes port **6333** (REST) and **6334** (gRPC).

If Docker is not running, we fall back to **in-memory** Qdrant — same API, no persistence.

In [27]:
import subprocess, time

QDRANT_PORT = 6333
COLLECTION  = "contracts"

def start_qdrant_docker():
    """Pull and start Qdrant container if Docker is available."""
    try:
        # Check if already running
        check = subprocess.run(
            ["docker", "ps", "--filter", "name=qdrant-legal", "--format", "{{.Names}}"],
            capture_output=True, text=True, timeout=10
        )
        if "qdrant-legal" in check.stdout:
            print("Qdrant container already running!")
            return True

        # Start container
        result = subprocess.run([
            "docker", "run", "-d", "--name", "qdrant-legal",
            "-p", f"{QDRANT_PORT}:6333",
            "-p", "6334:6334",
            "-v", f"{Path.cwd().parent}/data/qdrant_storage:/qdrant/storage",
            "qdrant/qdrant"
        ], capture_output=True, text=True, timeout=60)

        if result.returncode == 0:
            print("Qdrant container started! Waiting for it to be ready...")
            time.sleep(4)
            return True
        else:
            print(f"Docker start failed: {result.stderr[:200]}")
            return False
    except Exception as e:
        print(f"Docker not available: {e}")
        return False

docker_ok = start_qdrant_docker()

if docker_ok:
    client = QdrantClient(host="localhost", port=QDRANT_PORT)
    print(f"Connected to Qdrant at localhost:{QDRANT_PORT}")
else:
    print("Using in-memory Qdrant (no Docker)")
    client = QdrantClient(":memory:")

print(f"Qdrant version: {client.get_collections()}")

Qdrant container already running!
Connected to Qdrant at localhost:6333
Qdrant version: collections=[CollectionDescription(name='contracts')]


## 2 — Load Embedding Model

`all-MiniLM-L6-v2` produces **384-dimensional** vectors. We normalize them so cosine similarity == dot product.

In [28]:
print("Loading embedding model...")
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
VECTOR_DIM = 384
print(f"Embedding model loaded! Vector dim = {VECTOR_DIM}")

# Quick sanity check
test_vec = embedder.embed_query("indemnification clause")
print(f"Sample embedding shape: {len(test_vec)}, first 5 values: {test_vec[:5]}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded! Vector dim = 384
Sample embedding shape: 384, first 5 values: [0.011194621212780476, 0.11593766510486603, 0.04677082970738411, 0.014195934869349003, 0.0330585315823555]


## 3 — Create Qdrant Collection "contracts"

A **collection** in Qdrant is like a table in SQL — it holds vectors + their payloads (metadata).

We configure it with:
- `size=384` (matches MiniLM output)
- `distance=Cosine` (for semantic similarity)

In [29]:
from qdrant_client.models import VectorParams, Distance

# Delete old collection if exists (fresh start)
existing = [c.name for c in client.get_collections().collections]
if COLLECTION in existing:
    client.delete_collection(COLLECTION)
    print(f"Deleted existing collection: {COLLECTION}")

# Create fresh collection
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
)
print(f"Collection created: {COLLECTION}")
print(f"Collections: {[c.name for c in client.get_collections().collections]}")

Deleted existing collection: contracts
Collection created: contracts
Collections: ['contracts']


## 4 — Load Contracts + Assign Metadata

Each contract gets structured **payload** (metadata) stored alongside its vector:
- `contract_type` — e.g. Distributor, Endorsement, Consulting
- `jurisdiction` — inferred from contract text
- `upload_date` — simulated ingestion date

This enables **filtered search** later: e.g. *'only search Distributor agreements from 2020'*.

In [30]:
# Metadata mapping for our 10 sample contracts
METADATA_MAP = {
    "ADAMSGOLFINC":           {"contract_type": "Endorsement",   "jurisdiction": "US"},
    "CENTRACKINTERNATIONAL":  {"contract_type": "Hosting",       "jurisdiction": "US"},
    "DovaPharmaceuticals":    {"contract_type": "Promotion",     "jurisdiction": "US"},
    "KIROMICBIOPHARMA":       {"contract_type": "Consulting",    "jurisdiction": "US"},
    "LIMEENERGYCO":           {"contract_type": "Distributor",   "jurisdiction": "US"},
    "LohaCompany":            {"contract_type": "Supply",        "jurisdiction": "CN"},
    "NELNETINC":              {"contract_type": "Joint Filing",  "jurisdiction": "US"},
    "PACIRA":                 {"contract_type": "Licensing",     "jurisdiction": "US"},
    "VEONEER":                {"contract_type": "Joint Venture", "jurisdiction": "SE"},
    "WHITESMOKE":             {"contract_type": "Distribution",  "jurisdiction": "US"},
}

def get_metadata(filename):
    """Match filename prefix to metadata."""
    for key, meta in METADATA_MAP.items():
        if key.lower() in filename.lower():
            return meta
    return {"contract_type": "General", "jurisdiction": "US"}

contracts_dir = Path("../data/sample_contracts")
raw_docs = []
for fp in sorted(contracts_dir.glob("*.txt")):
    loader = TextLoader(str(fp), encoding="utf-8")
    docs = loader.load()
    meta = get_metadata(fp.name)
    for doc in docs:
        doc.metadata.update({
            "source_file":   fp.name,
            "contract_type": meta["contract_type"],
            "jurisdiction":  meta["jurisdiction"],
            "upload_date":   "2024-01-15",
        })
    raw_docs.extend(docs)
    print(f"  Loaded: {fp.name[:50]} | type={meta['contract_type']} | jur={meta['jurisdiction']}")

print(f"\nTotal documents: {len(raw_docs)}")
print(f"Total chars: {sum(len(d.page_content) for d in raw_docs):,}")

  Loaded: ADAMSGOLFINC_03_21_2005-EX-10_17-ENDORSEMENT_AGREE | type=Endorsement | jur=US
  Loaded: CENTRACKINTERNATIONALINC_10_29_1999-EX-10_3-WEB_SI | type=Hosting | jur=US
  Loaded: DovaPharmaceuticalsInc_20181108_10-Q_EX-10_2_11414 | type=Promotion | jur=US
  Loaded: KIROMICBIOPHARMA_INC_05_11_2020-EX-10_23-CONSULTIN | type=Consulting | jur=US
  Loaded: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR_AGREEMEN | type=Distributor | jur=US
  Loaded: LohaCompanyltd_20191209_F-1_EX-10_16_11917878_EX-1 | type=Supply | jur=CN
  Loaded: NELNETINC_04_08_2020-EX-1-JOINT_FILING_AGREEMENT.t | type=Joint Filing | jur=US
  Loaded: PACIRA_PHARMACEUTICALS__INC__-_A_R_STRATEGIC_LICEN | type=Licensing | jur=US
  Loaded: VEONEER_INC_02_21_2020-EX-10_11-JOINT_VENTURE_AGRE | type=Joint Venture | jur=SE
  Loaded: WHITESMOKE_INC_11_08_2011-EX-10_26-PROMOTION_AND_D | type=Distribution | jur=US

Total documents: 10
Total chars: 524,445


## 5 — Chunk with RecursiveCharacterTextSplitter

chunk_size=512, overlap=50 — same as Notebook 2. Metadata propagates to every child chunk.

In [31]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(raw_docs)
print(f"Chunks: {len(chunks)}")
print(f"Avg size: {sum(len(c.page_content) for c in chunks)//len(chunks)} chars")

# Show metadata is preserved
sample = chunks[5]
print(f"\nSample chunk metadata: {sample.metadata}")
print(f"Sample text: {sample.page_content[:200]}...")

Chunks: 1602
Avg size: 331 chars

Sample chunk metadata: {'source': '..\\data\\sample_contracts\\ADAMSGOLFINC_03_21_2005-EX-10_17-ENDORSEMENT_AGREEMENT.txt', 'source_file': 'ADAMSGOLFINC_03_21_2005-EX-10_17-ENDORSEMENT_AGREEMENT.txt', 'contract_type': 'Endorsement', 'jurisdiction': 'US', 'upload_date': '2024-01-15'}
Sample text: 1.[*****] 2.Sufficient [*****] to maintain total minimum of [*****] ADAMS GOLF  [*****] (includes [*****])[*****] at all times 3.[*****] 4.[*****] (CONSULTANT may continue to place the [*****] logo on...


## 6 — Embed Chunks & Index into Qdrant

We batch-embed all chunks and upsert them as **Points** into the `contracts` collection.
Each point has:
- `id` — integer
- `vector` — 384-float embedding
- `payload` — full metadata dict + the text itself

In [32]:
from qdrant_client.models import PointStruct
import uuid

print(f"Embedding and indexing {len(chunks)} chunks into Qdrant...")

BATCH = 64
points = []
for i, chunk in enumerate(chunks):
    vec = embedder.embed_query(chunk.page_content)
    points.append(PointStruct(
        id=i,
        vector=vec,
        payload={
            "text":          chunk.page_content,
            "source_file":   chunk.metadata.get("source_file", ""),
            "contract_type": chunk.metadata.get("contract_type", "General"),
            "jurisdiction":  chunk.metadata.get("jurisdiction", "US"),
            "upload_date":   chunk.metadata.get("upload_date", ""),
        }
    ))
    if (i+1) % BATCH == 0:
        client.upsert(collection_name=COLLECTION, points=points)
        points = []
        print(f"  Indexed {i+1}/{len(chunks)} chunks...")

# Flush remaining
if points:
    client.upsert(collection_name=COLLECTION, points=points)

info = client.get_collection(COLLECTION)
print(f"\nIndexing complete!")
print(f"Vectors in collection: {info.points_count}")

Embedding and indexing 1602 chunks into Qdrant...
  Indexed 64/1602 chunks...
  Indexed 128/1602 chunks...
  Indexed 192/1602 chunks...
  Indexed 256/1602 chunks...
  Indexed 320/1602 chunks...
  Indexed 384/1602 chunks...
  Indexed 448/1602 chunks...
  Indexed 512/1602 chunks...
  Indexed 576/1602 chunks...
  Indexed 640/1602 chunks...
  Indexed 704/1602 chunks...
  Indexed 768/1602 chunks...
  Indexed 832/1602 chunks...
  Indexed 896/1602 chunks...
  Indexed 960/1602 chunks...
  Indexed 1024/1602 chunks...
  Indexed 1088/1602 chunks...
  Indexed 1152/1602 chunks...
  Indexed 1216/1602 chunks...
  Indexed 1280/1602 chunks...
  Indexed 1344/1602 chunks...
  Indexed 1408/1602 chunks...
  Indexed 1472/1602 chunks...
  Indexed 1536/1602 chunks...
  Indexed 1600/1602 chunks...

Indexing complete!
Vectors in collection: 1602


## 7 — Load Local LLM (flan-t5-base)

`google/flan-t5-base` is a free, instruction-following model that runs locally on CPU.
It reads the retrieved contract chunks and generates a **cited answer**.

In [33]:
print("Loading LLM (flan-t5-base, ~1GB download on first run)...")
gen = hf_pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    do_sample=False,
)
llm = HuggingFacePipeline(pipeline=gen)
print("LLM loaded!")

# Quick test
resp = llm.invoke("What does 'indemnification' mean in a legal contract?")
print(f"\nLLM test answer: {resp}")

Loading LLM (flan-t5-base, ~1GB download on first run)...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

LLM loaded!

LLM test answer: What does 'indemnification' mean in a legal contract?


## 8 — Build the RAG QA Chain

**Flow:** Question → embed → Qdrant top-5 → inject into prompt → LLM → cited answer

The prompt explicitly instructs the model to cite which contract each piece of information came from.

In [34]:
# Build LangChain-compatible Qdrant retriever
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION,
    embedding=embedder,
    content_payload_key="text",      # which payload field holds the text
    metadata_payload_key="metadata", # (unused — we store flat)
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# Prompt template — instructs LLM to cite sources
PROMPT_TEMPLATE = """You are a legal contract analyst. Answer the question using ONLY the contract excerpts below.
For each fact, cite the contract in brackets like [Contract: filename].
If the answer is not in the excerpts, say "Not found in provided contracts."

CONTRACT EXCERPTS:
{context}

QUESTION: {question}

ANSWER (with citations):"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

print("RAG components ready!")

RAG components ready!


## 9 — Metadata Filtering

Qdrant supports **server-side filtering** — the similarity search only considers vectors whose payload matches the filter.
This is much faster than post-filtering because Qdrant never computes distances for non-matching points.

Filter fields available:
- `contract_type` — Distributor, Endorsement, Consulting, etc.
- `jurisdiction` — US, CN, SE
- `upload_date` — ISO date string

In [35]:
def search_with_filter(query: str, contract_type: str = None,
                       jurisdiction: str = None, k: int = 5) -> list[dict]:
    """
    Semantic search in Qdrant with optional metadata filters.
    Returns top-k results as dicts with text, score, and metadata.
    """
    query_vec = embedder.embed_query(query)

    # Build Qdrant filter
    must_conditions = []
    if contract_type:
        must_conditions.append(FieldCondition(
            key="contract_type",
            match=MatchValue(value=contract_type)
        ))
    if jurisdiction:
        must_conditions.append(FieldCondition(
            key="jurisdiction",
            match=MatchValue(value=jurisdiction)
        ))

    qdrant_filter = Filter(must=must_conditions) if must_conditions else None

    results = client.query_points(
        collection_name=COLLECTION,
        query=query_vec,
        query_filter=qdrant_filter,
        limit=k,
        with_payload=True,
    )

    return [{
        "score":         round(r.score, 4),
        "contract_type": r.payload.get("contract_type"),
        "jurisdiction":  r.payload.get("jurisdiction"),
        "source":        r.payload.get("source_file", "")[:50],
        "text":          r.payload.get("text", "")[:300],
    } for r in results.points]


# --- Demo 1: No filter ---
print("=" * 65)
print("SEARCH (no filter): 'indemnification clause'")
print("=" * 65)
for r in search_with_filter("indemnification clause", k=3):
    print(f"  [{r['score']}] [{r['contract_type']:15s}] {r['source']}")
    print(f"  {r['text'][:150]}...")
    print()

SEARCH (no filter): 'indemnification clause'
  [0.638] [Distributor    ] LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR_AGREEMEN
  5.3      Indemnification...

  [0.6349] [Licensing      ] PACIRA_PHARMACEUTICALS__INC__-_A_R_STRATEGIC_LICEN
  10.3 Conditions to Indemnification. Promptly after receipt by a Party of any Claim or alleged claim or notice of the commencement of any action, admin...

  [0.6311] [Licensing      ] PACIRA_PHARMACEUTICALS__INC__-_A_R_STRATEGIC_LICEN
  (b) the indemnifying Party will not, except with the consent of the indemnified Party (such consent not be unreasonably  withheld or delayed), consent...



In [36]:
# --- Demo 2: Filter by contract type ---
print("=" * 65)
print("SEARCH (Distributor only): 'termination notice period'")
print("=" * 65)
for r in search_with_filter("termination notice period", contract_type="Distributor", k=3):
    print(f"  [{r['score']}] [{r['contract_type']:15s}] {r['source']}")
    print(f"  {r['text'][:150]}...")
    print()

# --- Demo 3: Filter by jurisdiction ---
print("=" * 65)
print("SEARCH (US jurisdiction only): 'governing law'")
print("=" * 65)
for r in search_with_filter("governing law", jurisdiction="US", k=3):
    print(f"  [{r['score']}] [{r['jurisdiction']:5s}] {r['source']}")
    print(f"  {r['text'][:150]}...")
    print()

SEARCH (Distributor only): 'termination notice period'
  [0.5754] [Distributor    ] LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR_AGREEMEN
  4.2      Termination  for  Cause.   Either  party  may  terminate  this                   Agreement upon 30 days

                                    ...

  [0.4953] [Distributor    ] LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR_AGREEMEN
  4.1      Duration.   Unless  earlier   terminated   otherwise  provided                   therein,  this  Agreement,  subject to the  commencement  da...

  [0.4458] [Distributor    ] LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR_AGREEMEN
  .  The termination                   of this  Agreement  shall not relieve either party hereto from                   obligations  which have occurred...

SEARCH (US jurisdiction only): 'governing law'
  [0.5064] [US   ] DovaPharmaceuticalsInc_20181108_10-Q_EX-10_2_11414
  1.30 "Governmental Authority" shall mean any court, agency, authority, department, regulatory body or other instrum

## 10 — RAG QA Test

Now we run the complete RAG loop:
`Question → embed → Qdrant top-5 → LLM reads chunks → cited answer`

In [37]:
def ask(question: str):
    """Run the full RAG QA chain and print result with sources."""
    print(f"\nQUESTION: {question}")
    print("=" * 70)
    
    # 1. Retrieve
    docs = retriever.invoke(question)
    
    # 2. Format Context
    context = "\n\n".join(d.page_content for d in docs)
    
    # 3. Prompt LLM
    final_prompt = prompt.format(context=context, question=question)
    answer = llm.invoke(final_prompt)

    print(f"ANSWER:\n{answer}")
    print("\nSOURCE CHUNKS USED:")
    for i, doc in enumerate(docs, 1):
        src  = doc.metadata.get("source_file", "?")[:50]
        ctyp = doc.metadata.get("contract_type", "?")
        print(f"  [{i}] [{ctyp:15s}] {src}")
        print(f"       {doc.page_content[:120]}...")
    print()

# ── THE KEY TEST FROM THE SPEC ──
ask("Is this indemnification clause standard?")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors



QUESTION: Is this indemnification clause standard?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER:
You are a legal contract analyst. Answer the question using ONLY the contract excerpts below.
For each fact, cite the contract in brackets like [Contract: filename].
If the answer is not in the excerpts, say "Not found in provided contracts."

CONTRACT EXCERPTS:
11.3 Indemnification Procedures. The Party seeking indemnification under Section 11.1 or 11.2, as applicable (the "Indemnified Party") shall give prompt notice to the Party against whom indemnity is sought (the "Indemnifying Party") of the assertion or commencement of any Claim in respect of which indemnity may be sought under Section 11.1 or 11.2, as applicable, and will provide the Indemnifying Party such information with respect thereto that the Indemnifying Party may reasonably request

10.3 Conditions to Indemnification. Promptly after receipt by a Party of any Claim or alleged claim or notice of the commencement of any action, administrative or legal proceeding, or investigation as to which the indemnity provided 

In [38]:
ask("What is the termination notice period required in these contracts?")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What is the termination notice period required in these contracts?
ANSWER:
You are a legal contract analyst. Answer the question using ONLY the contract excerpts below.
For each fact, cite the contract in brackets like [Contract: filename].
If the answer is not in the excerpts, say "Not found in provided contracts."

CONTRACT EXCERPTS:
5. Term and Termination. This Agreement will commence on the Effective Date and will continue until termination as provided below.

Either Consultant or Company may terminate this Agreement upon prior written notice thereof to the other party.

Upon termination of this Agreement, all rights and duties of the parties hereunder shall cease except:

12.3 Other Early Termination.

12.3.1 Either Party shall have the right to terminate this Agreement before the end of the Term for its convenience upon [***] written notice to the other Party (and any such termination shall become effective at the end of such [***]); [***].

ARTICLE 12  TERM AND TERMI

In [39]:
ask("What governing law applies to the distributor agreement?")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What governing law applies to the distributor agreement?
ANSWER:
You are a legal contract analyst. Answer the question using ONLY the contract excerpts below.
For each fact, cite the contract in brackets like [Contract: filename].
If the answer is not in the excerpts, say "Not found in provided contracts."

CONTRACT EXCERPTS:
.  In addition,                   Company   agrees  to  indemnify,   defend  and  hold  harmless                   Distributor from and against all suits,  claims,  obligations,                   liabilities,   damages,   losses   and  the  like   (including                   attorneys'  fees  and  costs)  arising  out of or  related  to                   Company's manufacture or design of the Products, provided that                   Distributor  is not at fault in connection  with the same, and

(A)      During the Term of this Agreement and for a period of                            twelve (12) months  thereafter,  the  Distributor (on               

In [40]:
ask("Are there any exclusivity or non-compete restrictions?")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: Are there any exclusivity or non-compete restrictions?
ANSWER:
You are a legal contract analyst. Answer the question using ONLY the contract excerpts below.
For each fact, cite the contract in brackets like [Contract: filename].
If the answer is not in the excerpts, say "Not found in provided contracts."

CONTRACT EXCERPTS:
2.3 Non-Competition; Non-Solicitation.

.   4.16 Non-Compete. EKR shall not, during [**], market, distribute or sell a Competing Product in the Territory unless during such time an A/B  rated generic product of the Product(s) is launched in such country of the Territory or in the event this Agreement is terminated or EKR  exercises its rights under Section 17.4 hereof.   4.17 PPI as Exclusive Provider

. When endorsing a non-competitive product, under no circumstances shall CONSULTANT wear, play, use, hold or in any way be associated with an ADAMS GOLF competitor's Product.

-4-

"Competing Product"



Means any [**] ([**] hours) [**] preparation (other t

## 11 — Collection Info & Summary

In [41]:
from collections import Counter

info = client.get_collection(COLLECTION)
# Scroll through all payloads to get stats
scroll_res, _ = client.scroll(COLLECTION, limit=5000, with_payload=True)
types     = Counter(p.payload.get("contract_type","?") for p in scroll_res)
jurisdictions = Counter(p.payload.get("jurisdiction","?") for p in scroll_res)

print("=" * 55)
print("  QDRANT COLLECTION STATS")
print("=" * 55)
print(f"  Collection name  : {COLLECTION}")
print(f"  Total vectors    : {info.points_count}")
print(f"  Vector dimension : {info.config.params.vectors.size}")
print(f"  Distance metric  : {info.config.params.vectors.distance}")
print(f"\n  Contract types:")
for ct, cnt in types.most_common():
    print(f"    {ct:20s}: {cnt} chunks")
print(f"\n  Jurisdictions:")
for jur, cnt in jurisdictions.most_common():
    print(f"    {jur:10s}: {cnt} chunks")
print("=" * 55)

  QDRANT COLLECTION STATS
  Collection name  : contracts
  Total vectors    : 1602
  Vector dimension : 384
  Distance metric  : Cosine

  Contract types:
    Promotion           : 533 chunks
    Licensing           : 443 chunks
    Distribution        : 218 chunks
    Distributor         : 175 chunks
    Endorsement         : 75 chunks
    Consulting          : 54 chunks
    Hosting             : 41 chunks
    Supply              : 36 chunks
    Joint Venture       : 23 chunks
    Joint Filing        : 4 chunks

  Jurisdictions:
    US        : 1543 chunks
    CN        : 36 chunks
    SE        : 23 chunks


---
✅ **Done!** Qdrant collection `contracts` is live with metadata filtering and a full RAG QA chain.